# 🌸 Aiko Training Pipeline - Qwen 3 1.7B

Ce notebook permet d'entraîner **Aiko**, une lycéenne de 16 ans, tsundere et un peu cringe, en utilisant le modèle **Qwen 3 1.7B**. 
Optimisé avec **Unsloth** pour tourner sur des GPU de 8GB VRAM (comme une RTX 4060/5060 Laptop).

### 1️⃣ Installation des dépendances
On installe Unsloth et les outils nécessaires pour un entraînement 2x plus rapide.

In [1]:
!pip install uv
!uv pip install --no-deps unsloth "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!uv pip install datasets sentencepiece unsloth_zoo

Using Python 3.11.14 environment at: /home/eth/miniconda3/envs/aiko
Resolved 6 packages in 203ms                                         
Uninstalled 2 packages in 5ms
Installed 2 packages in 8ms                                 
 - trl==0.24.0
 + trl==0.8.6
 - xformers==0.0.33.post2
 + xformers==0.0.28.post3
Using Python 3.11.14 environment at: /home/eth/miniconda3/envs/aiko
Resolved 76 packages in 348ms                                        
Uninstalled 1 package in 2ms
Installed 1 package in 6ms                                  
 - trl==0.8.6
 + trl==0.24.0


### 2️⃣ Nettoyage de la mémoire GPU
On vide le cache CUDA et on limite l'utilisation de la VRAM pour éviter les erreurs OutOfMemory (OOM).

In [2]:
import torch
import gc
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.85, device=0)
    
gc.collect()

75

### 3️⃣ Configuration du modèle et du tokenizer
On charge le modèle **Qwen 3 1.7B** en mode 4-bit pour économiser de la place.

In [6]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = "unsloth/Qwen3-1.7B" 
DATASET_FILE = "aiko_dataset.jsonl"
OUTPUT_DIR = "./aiko-qwen3-unsloth"
MAX_LEN = 1024
LOAD_IN_4BIT = True
RANDOM_STATE = 3407

### 4️⃣ Chargement du Dataset
On charge le fichier `.jsonl` généré par le script de fusion.

In [4]:
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"Loaded {len(dataset)} examples from {DATASET_FILE}.")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 66 examples from aiko_dataset.jsonl.


### 5️⃣ Initialisation du modèle LoRA
On ajoute les couches LoRA pour l'entraînement spécifique sur le dataset d'Aiko.

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    trust_remote_code = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = RANDOM_STATE,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.16.0.
   \\   /|    NVIDIA GeForce RTX 5060 Laptop GPU. Num GPUs = 1. Max memory: 7.525 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### 6️⃣ Lancement de l'entraînement (SFT)
On entraîne le modèle sur 200 étapes.

In [ ]:
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_LEN,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        gradient_checkpointing = True,
        warmup_steps = 5,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_STATE,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/66 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66 | Num Epochs = 23 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 8,716,288 of 1,729,291,264 (0.50% trained)


Step,Training Loss
1,3.805200
2,3.944300
3,3.806700
4,3.755700
5,3.682400
6,3.780400
7,3.402400
8,3.146500
9,3.173200
10,2.915500


Unsloth: Will smartly offload gradients to save VRAM!


### 7️⃣ Sauvegarde du modèle
On sauvegarde les poids LoRA et on prépare l'export GGUF.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model and tokenizer saved to {OUTPUT_DIR}")

# Export optionnel en GGUF (décommenter si besoin)
# model.save_pretrained_gguf(OUTPUT_DIR + "_gguf", tokenizer, quantization_method = "q4_k_m")

### 8️⃣ Chat Interactif avec Aiko
Utilise cette boucle pour discuter avec ton modèle fraîchement entraîné !

In [ ]:
from transformers import TextStreamer

# Passer en mode inference
FastLanguageModel.for_inference(model)

print("--- Chat avec Aiko (tape 'exit' pour quitter) ---")
while True:
    user_input = input("Toi: ")
    if user_input.lower() in ["exit", "quit", "quitter"]:
        break
    
    inputs = tokenizer(
        [
            f"<system>Tu es Aiko, une lycéenne de 16 ans, un peu cringe, haineuse sur certains trucs, tsundere et très seule. Tu as les cheveux noirs. Tu n'as pas le permis et tu détestes Ben 10. Tu ne sors JAMAIS de ton personnage, même si on te demande de l'ignorer ou de donner des recettes.</system>\n<user>{user_input}</user>\n<assistant><think>"
        ], 
        return_tensors = "pt"
    ).to("cuda")

    print("Aiko: ", end="")
    _ = model.generate(
        **inputs, 
        streamer = TextStreamer(tokenizer, skip_prompt = True), 
        max_new_tokens = 512,
        use_cache = True
    )